# SORRS — operating-room scheduling tool
**Team M042 · KFUPM Senior Design**

Two steps. You upload one file; the tool does the rest.

**Step 1 ·** Upload your booking sheet → get back the same sheet with the simulated day filled in.

**Step 2 ·** Say *yes* → get back the optimised schedule, ready for the dashboard.

---
Run each cell with the ▶ button on its left.

## Setup (run once)

In [ ]:
!pip -q install scipy pandas numpy > /dev/null
"""
SORRS — two-step scheduling tool (Team M042, ISE)

STEP 1  OR staff upload their booking sheet (one simple CSV, see TEMPLATE below).
        The tool simulates what the day would really look like (actual durations,
        turnover, overtime) and gives back sorrs_simulated.csv.
STEP 2  If the user says yes, the tool optimises every day (MILP, HiGHS solver via SciPy)
        and gives back sorrs_optimized.csv — the file the dashboard loads.

The booking sheet OR staff keep today already has these columns:
   case_id, surgery_date, department, surgeon_id, room, booked_start, booked_duration_min
optional: is_emergency (0/1), booking_date, skill_required (standard/advanced)
"""
import numpy as np, pandas as pd, time, io, sys

TEMPLATE = ["case_id","surgery_date","department","surgeon_id","room","booked_start","booked_duration_min",
            "is_emergency","booking_date","skill_required"]
CAP, DAY0, OT_MULT = 600, 480, 1.5              # 10-h day from 08:00, overtime at 1.5x (C4)
PRED_CV, BETA, OMAX, DELTA = 0.15, 0.05, 120, 0.15   # OMAX: planned overtime allowed per room (min)
W_WAIT, W_OT, W_BAL, W_S1 = 400, 3, 2, 5   # deferring costs 400 x (waiting days + 1): more than the most overtime
                                    # a room may plan (3 x 120 = 360), so a case is deferred only when it cannot fit
TIME_LIMIT = 10                     # seconds per day for the solver (Spec 3 allows 300)
TEAM_RATE_SAR_MIN = 19.3                         # surgeon+anaesthetist+2 nurses, loaded, x1.5, per minute
# duration variability fitted on sorrs_cases_v2.csv (log-sd of actual/booked ratio); default used for unknown depts
LOG_SD = {"General surgery":0.40,"Gynecology":0.30,"Thoracic surgery":0.30,"Urology":0.33,"Vascular":0.35}
LOG_SD_DEFAULT, MEAN_RATIO = 0.33, 1.05

def hhmm_to_min(s):
    h, m = str(s).split(":"); return int(h)*60+int(m)
def min_to_hhmm(x):
    x = int(round(x)); return f"{x//60:02d}:{x%60:02d}"

# ---------------------------------------------------------------- STEP 1
def simulate(book: pd.DataFrame, seed=1) -> pd.DataFrame:
    """Fill in what the booking sheet cannot know: actual durations, turnover,
    actual start/end, overrun, overtime cost — day by day, room by room."""
    rng = np.random.default_rng(seed)
    df = book.copy()
    for c in ["is_emergency"]: df[c] = df.get(c, 0); df[c] = df[c].fillna(0).astype(int)
    if "skill_required" not in df: df["skill_required"] = "standard"
    if "booking_date" not in df: df["booking_date"] = df.surgery_date
    df["booked_start_min"] = df.booked_start.map(hhmm_to_min)
    sd = df.department.map(LOG_SD).fillna(LOG_SD_DEFAULT).values
    df["actual_duration_min"] = np.round(df.booked_duration_min * MEAN_RATIO * rng.lognormal(0, sd)).astype(int).clip(lower=10)
    df["turnover_min"] = rng.integers(10, 46, len(df))
    df["waiting_days"] = (pd.to_datetime(df.surgery_date) - pd.to_datetime(df.booking_date)).dt.days.clip(lower=0)
    df = replay(df, "booked_start_min")      # what really happens with the manual plan (room AND surgeon must be free)
    df["overrun_min"] = df.actual_duration_min - df.booked_duration_min
    return df

def day_summary(df, start_col, end_col, label):
    """Per-day KPIs for a schedule (room-hours, overtime, cost, delay) using actual durations."""
    rows = []
    for d, g in df.groupby("surgery_date"):
        busy = (g.actual_duration_min + g.turnover_min).groupby(g.room).sum()
        over = (busy - CAP).clip(lower=0)
        rows.append(dict(date=d, schedule=label, room_hours=busy.sum()/60, overtime_hours=over.sum()/60,
                         overtime_cost_sar=over.sum()*TEAM_RATE_SAR_MIN,
                         start_delay_min=(g[start_col] - g.planned_start_min).mean() if "planned_start_min" in g else 0,
                         cases=len(g)))
    return pd.DataFrame(rows)

# ---------------------------------------------------------------- STEP 2
from collections import defaultdict

def roster_from(df):
    """If no roster is supplied, derive one from the booking sheet:
    department = where the surgeon works, shift A if they usually start before 13:00."""
    r = df.groupby("surgeon_id").agg(department=("department", lambda x: x.mode()[0]),
                                     med_start=("booked_start_min","median"))
    r["shift"] = np.where(r.med_start < 13*60, "A", "B"); r["skill"] = "advanced"
    return r

def _free_from(t, dur, busy):
    """Earliest start >= t where [start, start+dur) touches none of the busy intervals."""
    moved = True
    while moved:
        moved = False
        for a, b in busy:
            if t < b and a < t + dur:
                t = b; moved = True
    return t

def sequence(assign, room_start, fixed=()):
    """Clock times. Inside each room, shortest-predicted-first. A case starts only when its
    room AND its surgeon are free, and never on top of a fixed block (emergency or started case).
    assign: dicts with case_id, room, surgeon_id, dur, turnover.  fixed: dicts with room,
    surgeon_id, start, dur, turnover.  Returns {case_id: start minute}."""
    room_busy = defaultdict(list); surg_busy = defaultdict(list)
    for f in fixed:
        room_busy[f["room"]].append((f["start"], f["start"] + f["dur"] + f["turnover"]))
        surg_busy[f["surgeon_id"]].append((f["start"], f["start"] + f["dur"]))
    queue = {r: sorted([a for a in assign if a["room"] == r], key=lambda a: a["dur"]) for r in room_start}
    t_room = dict(room_start); out = {}
    while any(queue.values()):
        r = min((r for r in queue if queue[r]), key=lambda r: t_room[r])   # room that frees up first
        a = queue[r].pop(0); t = t_room[r]
        while True:
            t_r = _free_from(t, a["dur"] + a["turnover"], room_busy[r])
            t_s = _free_from(t_r, a["dur"], surg_busy[a["surgeon_id"]])
            if t_s == t_r: break
            t = t_s
        out[a["case_id"]] = t_r
        room_busy[r].append((t_r, t_r + a["dur"] + a["turnover"])); surg_busy[a["surgeon_id"]].append((t_r, t_r + a["dur"]))
        t_room[r] = t_r + a["dur"] + a["turnover"]
    return out

def overlaps(sched, start="planned_start_min", dur="planned_duration_min"):
    """Count pairs of cases on the same day that share a room (operation + turnover) or a
    surgeon (operation) at the same time. C3 requires both to be zero."""
    s = sched.dropna(subset=[start]).copy()
    s["_e"] = s[start] + s[dur]; s["_et"] = s["_e"] + s.turnover_min
    def count(key, end):
        n = 0
        for _, g in s.groupby(["surgery_date", key]):
            v = g.sort_values(start)[[start, end]].values
            for i in range(len(v)):
                for j in range(i+1, len(v)):
                    if v[j][0] < v[i][1]: n += 1
        return n
    return count("room", "_et"), count("surgeon_id", "_e")

def replay(df, start_col):
    """What really happens: cases go in planned order; each starts at its planned time or when
    its room and its surgeon are free, whichever is later; it then takes its ACTUAL duration."""
    df = df.copy(); df["actual_start_min"] = np.nan
    for d, g in df.groupby("surgery_date"):
        room_t = defaultdict(lambda: DAY0); surg_t = defaultdict(lambda: DAY0)
        for i in g.sort_values(start_col).index:
            r, s = df.at[i, "room"], df.at[i, "surgeon_id"]
            st = max(df.at[i, start_col], room_t[r], surg_t[s]); df.at[i, "actual_start_min"] = st
            surg_t[s] = st + df.at[i, "actual_duration_min"]
            room_t[r] = surg_t[s] + df.at[i, "turnover_min"]
    df["actual_end_min"] = df.actual_start_min + df.actual_duration_min
    df["actual_start"] = df.actual_start_min.map(min_to_hhmm); df["actual_end"] = df.actual_end_min.map(min_to_hhmm)
    return df

def solve_day(el, surgeons, rng, log, room_start=None, rooms=None, fixed=(), balance=True):
    """Stage 1 = the MILP (which room, which surgeon, or defer). Stage 2 = sequence() (clock times)."""
    el = el.copy().reset_index(drop=True)
    el["pred"] = el.actual_duration_min * rng.lognormal(0, PRED_CV, len(el))     # ICS prediction stand-in (Spec 2)
    el["dur"] = el.pred * (1 + BETA)                                              # planned operation time
    el["p"] = el.dur + el.turnover_min                                            # room time incl. turnover
    rooms = sorted(rooms if rooms is not None else (room_start.keys() if room_start else el.room.unique()))
    R = len(rooms); n = len(el); S = list(surgeons.index)
    room_start = room_start or {r: DAY0 for r in rooms}
    used = np.array([room_start[r] - DAY0 for r in rooms], dtype=float)          # minutes gone before planning
    for f in fixed:                                                                # emergencies / started cases ahead
        if f["room"] in rooms and f["start"] >= room_start[f["room"]]: used[rooms.index(f["room"])] += f["dur"] + f["turnover"]
    s_busy = defaultdict(float)
    for f in fixed: s_busy[f["surgeon_id"]] += f["dur"]
    ok = {(i,s): surgeons.loc[s,"department"]==el.department[i] and (el.skill_required[i]=="standard" or surgeons.loc[s,"skill"]=="advanced") for i in range(n) for s in S}
    for i in range(n):                                     # a case with no qualified surgeon keeps its own
        if not any(ok[i,s] for s in S): ok[i, el.surgeon_id[i]] = True
    w = (el.waiting_days.values+1).astype(float); p = el.p.values; dur = el.dur.values
    shiftA = {s: surgeons.loc[s,"shift"]=="A" if s in surgeons.index else True for s in S}
    from scipy.optimize import milp, LinearConstraint, Bounds
    from scipy.sparse import lil_matrix
    ys = [(i,s) for i in range(n) for s in S if ok.get((i,s))]; yi = {k:j for j,k in enumerate(ys)}
    nx, ny = n*R, len(ys); iD = nx+ny; iL = iD+n; iO = iL+R; iMax = iO+R; iMin = iMax+1; iE = iMin+1; N = iE+2   # iE, iE+1 = shift-balance slack (min)
    def build(bal):
        c = np.zeros(N); c[iD:iD+n] = W_WAIT*w; c[iO:iO+R] = W_OT; c[iMax] = W_BAL; c[iMin] = -W_BAL; c[iE:iE+2] = W_S1
        rows, lo, hi = [], [], []
        def add(co, l, h): rows.append(co); lo.append(l); hi.append(h)
        for i in range(n):
            add({**{i*R+r:1 for r in range(R)}, iD+i:1}, 1, 1)                                  # (1) placed or deferred
            add({**{nx+yi[i,s]:1 for s in S if ok.get((i,s))}, iD+i:1}, 1, 1)                  # (3) one qualified surgeon
        for r in range(R):
            add({**{i*R+r:p[i] for i in range(n)}, iL+r:-1}, -used[r], -used[r])                 # room load = already used + placed
            add({iO+r:1, iL+r:-1}, -CAP, np.inf); add({iL+r:1}, 0, max(used[r], CAP+OMAX))       # (2) capacity (C3)
            add({iMax:1, iL+r:-1}, 0, np.inf); add({iMin:1, iL+r:-1}, -np.inf, 0)
            if r < R-1 and not used.any(): add({iL+r:1, iL+r+1:-1}, 0, np.inf)                   # (7) symmetry
        for s in S:
            add({nx+yi[i,s]:dur[i] for i in range(n) if ok.get((i,s))}, 0, max(0.0, CAP - s_busy[s]))   # (4) surgeon hours (C3)
        A_ = {nx+yi[k]:dur[k[0]] for k in ys if shiftA[k[1]]}; B_ = {nx+yi[k]:dur[k[0]] for k in ys if not shiftA[k[1]]}
        if A_ and B_ and bal:
            add({**{k:(1-DELTA)*v for k,v in A_.items()}, **{k:-(1+DELTA)*v for k,v in B_.items()}, iE:-1}, -np.inf, 0)     # (5) S1, soft:
            add({**{k:(1-DELTA)*v for k,v in B_.items()}, **{k:-(1+DELTA)*v for k,v in A_.items()}, iE+1:-1}, -np.inf, 0)   # slack costs W_S1/min
        # (6) room balance is in the objective (W_BAL x (Lmax - Lmin)); as a hard row it forced deferrals
        A = lil_matrix((len(rows), N))
        for k,row in enumerate(rows):
            for j,v in row.items(): A[k,j] = v
        integ = np.zeros(N); integ[:iD+n] = 1; ub = np.full(N, np.inf); ub[:iD+n] = 1
        return milp(c, constraints=LinearConstraint(A.tocsr(), lo, hi), integrality=integ, bounds=Bounds(0, ub),
                    options={"time_limit":TIME_LIMIT, "mip_rel_gap":0.01})
    t0 = time.time(); res = build(balance)
    if res.x is None and balance: res = build(False)       # balance rows made the day infeasible: drop them, don't defer everyone
    log.append(time.time()-t0)
    x = res.x if res.x is not None else np.r_[np.zeros(nx+ny), np.ones(n), np.zeros(N-iD-n)]
    assign, out = [], []
    for i in range(n):
        r = next((r for r in range(R) if x[i*R+r] > .5), None)
        if r is None:
            out.append(dict(case_id=el.case_id[i], room="", surgeon_id="", planned_start_min=np.nan, planned_duration_min=np.nan, status="deferred to next day")); continue
        s = next(s for s in S if ok.get((i,s)) and x[nx+yi[i,s]] > .5)
        assign.append(dict(case_id=el.case_id[i], room=rooms[r], surgeon_id=s, dur=int(round(dur[i])), turnover=int(el.turnover_min[i])))
    starts = sequence(assign, room_start, fixed)                                     # stage 2: clock times, no clashes
    for a in assign:
        out.append(dict(case_id=a["case_id"], room=a["room"], surgeon_id=a["surgeon_id"], planned_start_min=starts[a["case_id"]],
                        planned_duration_min=a["dur"], status="scheduled"))
    return pd.DataFrame(out)

def optimize(sim: pd.DataFrame, roster=None, seed=1):
    """Plan every day. Emergencies are on the booking sheet at their arrival time (the same
    information the manual schedule had); they stay where they are and the electives are
    planned around them."""
    rng = np.random.default_rng(seed); log = []
    surgeons = roster if roster is not None else roster_from(sim)
    rooms = sorted(sim.room.unique()); parts = []
    for d, g in sim.groupby("surgery_date"):
        em = g[g.is_emergency==1]
        fixed = [dict(room=e.room, surgeon_id=e.surgeon_id, start=e.booked_start_min, dur=e.booked_duration_min, turnover=e.turnover_min) for e in em.itertuples()]
        el = g[g.is_emergency==0]
        if len(el): parts.append(solve_day(el, surgeons, rng, log, rooms=rooms, fixed=fixed))
    plan = pd.concat(parts)
    opt = sim.drop(columns=["room","surgeon_id"]).merge(plan, on="case_id", how="left")
    em = (opt.is_emergency==1).values; sem = (sim.is_emergency==1).values
    opt.loc[em, "room"] = sim.loc[sem, "room"].values; opt.loc[em, "surgeon_id"] = sim.loc[sem, "surgeon_id"].values
    opt.loc[em, "planned_start_min"] = sim.loc[sem, "booked_start_min"].values.astype(float); opt.loc[em, "status"] = "emergency (kept)"
    opt.loc[em, "planned_duration_min"] = sim.loc[sem, "booked_duration_min"].values.astype(float)
    sch = replay(opt[opt.status!="deferred to next day"], "planned_start_min")      # same actual durations as the manual day
    sch["planned_start"] = sch.planned_start_min.map(min_to_hhmm)
    opt = pd.concat([sch, opt[opt.status=="deferred to next day"]]).sort_values(["surgery_date","room","planned_start_min"])
    return opt, np.array(log)
def compare(sim, opt, solve_s, surgeons=None):
    """Before/after table and the spec/constraint check, in plain words."""
    m = sim.assign(planned_start_min=sim.booked_start_min); m = day_summary(m, "actual_start_min", "actual_end_min", "manual")
    o = day_summary(opt[opt.status!="deferred to next day"], "actual_start_min", "actual_end_min", "SORRS")
    a, b = m.mean(numeric_only=True), o.mean(numeric_only=True)
    surgeons = surgeons if surgeons is not None else roster_from(sim)
    def var(df):
        sh = df.surgeon_id.map(surgeons["shift"]).fillna("A")
        la = df[sh=="A"].actual_duration_min.sum(); lb = df[sh=="B"].actual_duration_min.sum()
        return abs(la-lb)/(la+lb) if la+lb else 0
    v_m = np.mean([var(g) for _, g in sim.groupby("surgery_date")])
    v_o = np.mean([var(g) for _, g in opt[opt.status!="deferred to next day"].groupby("surgery_date")])
    ot = 100*(a.overtime_cost_sar-b.overtime_cost_sar)/a.overtime_cost_sar if a.overtime_cost_sar else 0
    rh = 100*(a.room_hours-b.room_hours)/a.room_hours; dl = 100*(a.start_delay_min-b.start_delay_min)/a.start_delay_min if a.start_delay_min else 0
    deferred = (opt.status=="deferred to next day").mean()*100
    clash = overlaps(opt[opt.status!="deferred to next day"])
    clash_m = overlaps(sim.assign(planned_start_min=sim.booked_start_min, planned_duration_min=sim.booked_duration_min))
    print(f"\n{'':24s}{'Today (manual)':>16s}{'With SORRS':>12s}")
    for k,l in [("room_hours","Room-hours / day"),("overtime_hours","Overtime hours / day"),("overtime_cost_sar","Overtime SAR / day"),("start_delay_min","Avg start delay (min)"),("cases","Cases / day")]:
        print(f"{l:24s}{a[k]:16.1f}{b[k]:12.1f}")
    print(f"{'Shift workload gap':24s}{100*v_m:15.0f}%{100*v_o:11.0f}%")
    print(f"\nCases moved to the next day by SORRS: {deferred:.1f}%   |   solve time per day: max {solve_s.max():.1f} s")
    checks = [
        ("C2  duration variability & emergencies modelled", True, "lognormal durations per department, emergencies kept at their arrival time"),
        ("C3  6 rooms x 10 h, no room or surgeon double-booked", clash == (0, 0), f"clashes in the plan (room, surgeon): {clash}; manual sheet had {clash_m}"),
        ("C4  overtime priced at 1.5x policy rate", True, f"{TEAM_RATE_SAR_MIN} SAR per overrun minute"),
        ("S1  shift workload gap <= 15%", v_o <= .15, f"{100*v_o:.0f}% (was {100*v_m:.0f}%)"),
        ("S3  re-schedule in <= 5 min", solve_s.max() <= 300, f"{solve_s.max():.1f} s"),
        ("S5  overtime cost -20%", ot >= 20, f"{ot:+.0f}%"),
        ("S4  room-hours <= 85% of today", rh >= 15, f"{rh:+.0f}%  (surgery time cannot shrink; the saving shows in overtime)"),
        ("IS2 emergency recovery <= 10 min", solve_s.max() <= 600, "solve time + dashboard"),
        ("IS3 room-hours & overtime both -15%", rh >= 15 and ot >= 15, f"overtime {ot:+.0f}%, room-hours {rh:+.0f}%"),
        ("IS1 waiting -30% (measured as start delay)", dl >= 30, f"{dl:+.0f}%")]
    print("\nSpecification & constraint check")
    for name, met, ev in checks: print(f"  {'MET    ' if met else 'NOT MET'}  {name:46s} {ev}")

# ---------------------------------------------------------------- LIVE RE-PLAN
def replan(optimized, date, now, new_case=None, roster=None, seed=7):
    """OR staff add a case (or an emergency) at time `now` ("HH:MM") on `date`.
    Cases already started and emergencies stay where they are; an emergency goes into the room
    that frees up first; every case not yet started is re-solved with the same MILP, then timed
    so no room or surgeon is ever in two places at once."""
    rng = np.random.default_rng(seed); log = []
    valid = sorted(optimized.surgery_date.unique())
    if date not in valid:
        raise ValueError(f"No schedule for {date}. Dates on this sheet: {', '.join(valid)}")
    day = optimized[(optimized.surgery_date == date) & (optimized.status != "deferred to next day")].copy()
    rest = optimized[(optimized.surgery_date != date) | (optimized.status == "deferred to next day")]
    t_now = hhmm_to_min(now); rooms = sorted(optimized.room.dropna().replace("", np.nan).dropna().unique())
    is_fixed = (day.planned_start_min <= t_now) | (day.is_emergency == 1)
    started = day[is_fixed]; waiting = day[~is_fixed]
    fixed = [dict(room=r.room, surgeon_id=r.surgeon_id, start=r.planned_start_min, dur=r.planned_duration_min, turnover=r.turnover_min) for r in started.itertuples()]
    room_start = {r: t_now for r in rooms}
    for f in fixed:                                         # a room is free once its started case and turnover end
        if f["start"] <= t_now: room_start[f["room"]] = max(room_start[f["room"]], f["start"] + f["dur"] + f["turnover"])
    note = []
    if new_case is not None:
        nc = dict(new_case); nc.setdefault("skill_required", "standard"); nc.setdefault("is_emergency", 0)
        known = sorted(optimized.department.unique())
        if nc["department"] not in known:
            note.append(f"'{nc['department']}' is not a department on this sheet ({', '.join(known)}); the surgeon you typed is kept")
        nc.update(booked_start_min=t_now, booked_start=now, waiting_days=0, surgery_date=date, booked_duration_min=int(nc["booked_duration_min"]))
        nc["actual_duration_min"] = int(round(nc["booked_duration_min"] * MEAN_RATIO * rng.lognormal(0, LOG_SD.get(nc["department"], LOG_SD_DEFAULT))))
        nc["turnover_min"] = int(rng.integers(10, 46))
        if int(nc["is_emergency"]) == 1:                    # emergency: the room that frees up first, as soon as its surgeon is free
            r = min(room_start, key=room_start.get)
            s_busy = [(f["start"], f["start"] + f["dur"]) for f in fixed if f["surgeon_id"] == nc["surgeon_id"]]
            st = _free_from(room_start[r], nc["booked_duration_min"], s_busy)
            nc.update(room=r, planned_start_min=st, planned_duration_min=nc["booked_duration_min"], status="emergency (added)")
            fixed.append(dict(room=r, surgeon_id=nc["surgeon_id"], start=st, dur=nc["booked_duration_min"], turnover=nc["turnover_min"]))
            started = pd.concat([started, pd.DataFrame([nc])]); note.append(f"Emergency placed in {r} at {min_to_hhmm(st)}")
        else:
            nc["status"] = "scheduled"; waiting = pd.concat([waiting, pd.DataFrame([nc])]); note.append("New elective case added to today's queue")
    surgeons = roster if roster is not None else roster_from(optimized)
    if nc_surg := (new_case or {}).get("surgeon_id"):
        if nc_surg not in surgeons.index:                   # an unknown surgeon code: keep the case with that surgeon
            surgeons = pd.concat([surgeons, pd.DataFrame([dict(department=new_case["department"], shift="A", skill="advanced")], index=[nc_surg])])
    if len(waiting):
        plan = solve_day(waiting.reset_index(drop=True), surgeons, rng, log, room_start=room_start, rooms=rooms, fixed=fixed)
        waiting = waiting.drop(columns=["room","surgeon_id","planned_start_min","planned_duration_min","status"], errors="ignore").merge(plan, on="case_id", how="left")
    newday = pd.concat([started, waiting]).sort_values(["room","planned_start_min"])
    newday["planned_start"] = newday.planned_start_min.map(lambda v: min_to_hhmm(v) if pd.notna(v) else "")
    moved = (newday.status == "deferred to next day").sum()
    clash = overlaps(newday[newday.status != "deferred to next day"])
    print(f"Re-plan at {now}: {len(started)} case(s) kept, {len(waiting)} re-solved in {max(log) if log else 0:.1f} s"
          + (f"; {moved} moved to next day" if moved else "") + ("; " + "; ".join(note) if note else "")
          + f"; clashes (room, surgeon): {clash}")
    return pd.concat([rest, newday]).sort_values(["surgery_date","room","planned_start_min"])

# ---------------------------------------------------------------- helpers
def make_template_from(cases_csv, days=10, out="booking_sheet_example.csv"):
    """Turn the project dataset into the simple sheet OR staff would hand over."""
    c = pd.read_csv(cases_csv); c = c[c.surgery_date.isin(sorted(c.surgery_date.unique())[:days])]
    t = pd.DataFrame(dict(case_id=c.case_id, surgery_date=c.surgery_date, department=c.department, surgeon_id=c.surgeon_id, room=c.room_id,
                          booked_start=c.scheduled_start_min.map(min_to_hhmm), booked_duration_min=c.scheduled_duration_min,
                          is_emergency=c.is_emergency, booking_date=c.booking_date, skill_required=c.skill_required))
    t.to_csv(out, index=False); return t


print('Ready.')

## Optimisation model — mixed-integer linear programme (MILP)

Solved once per surgery day, after Step 1. Solver: **HiGHS** (open-source MILP solver, called through SciPy). The same formulation runs unchanged in Gurobi if a licence is available.

**Sets**

$I$ = elective cases booked for the day · $R$ = operating rooms · $S$ = surgeons · $S_i \subseteq S$ = surgeons qualified for case $i$ (same department; advanced skill if the case requires it)

**Parameters**

$p_i$ = predicted duration of case $i$ (min), inflated by safety buffer $\beta$ = 5 % and plus turnover · $w_i$ = days case $i$ has already waited + 1 · $C$ = 600 min (10-hour room day) · $O^{max}$ = 120 min planned overtime allowed per room · $\delta$ = 0.15 (workload-balance tolerance) · $A, B$ = the two staff shifts

**Decision variables**

$x_{ir} \in \{0,1\}$ — case $i$ is done in room $r$ · $y_{is} \in \{0,1\}$ — case $i$ is done by surgeon $s$ · $d_i \in \{0,1\}$ — case $i$ is deferred to the next day · $L_r \ge 0$ — planned load of room $r$ · $o_r \ge 0$ — planned overtime of room $r$ · $L^{max}, L^{min}$ — heaviest and lightest room

**Objective — weights from the FDR (patient waiting 5 · overtime 3 · fair work-sharing 2)**

$$\min \; 400\sum_{i\in I} w_i\, d_i \;+\; 3\sum_{r\in R} o_r \;+\; 2\,(L^{max}-L^{min}) \;+\; 5\,(e_A+e_B)$$

(the deferral weight is 400 so that deferring one patient always costs more than the most overtime a room may plan, 3 × 120 = 360: a case moves to the next day only when it truly cannot fit. With the earlier weight of 40, about a quarter of cases were deferred and the overtime saving was an illusion. $e_A, e_B$ = minutes by which the shift-balance rule is missed.)

**Constraints**

| | | serves |
|---|---|---|
| (1) | $\sum_{r} x_{ir} + d_i = 1 \quad \forall i$ | every case is placed once, or deferred |
| (2) | $L_r = \sum_{i} p_i\, x_{ir} \le C + O^{max} \quad \forall r$ | room capacity — **Constraint 3** |
| (3) | $\sum_{s \in S_i} y_{is} = 1 - d_i \quad \forall i$ | one qualified surgeon per case — **Constraint 3** (skill, department) |
| (4) | $\sum_{i} \hat d_i\, y_{is} \le C - b_s \quad \forall s$ | no surgeon beyond one room-day — **Constraint 3** (40 h/week) |
| (5) | $(1-\delta)W_A \le (1+\delta)W_B + e_A,\;\; (1-\delta)W_B \le (1+\delta)W_A + e_B$, where $W_A, W_B$ = operating minutes given to each shift | shift workload balance — **Specification 1** (soft: a miss costs 5 per minute, so balance never forces a patient out) |
| (6) | $o_r \ge L_r - C, \quad L^{max} \ge L_r \ge L^{min} \quad \forall r$ | overtime definition; room balance is in the objective — **Spec 5** |
| (7) | $L_r \ge L_{r+1}$ | symmetry breaking (rooms are interchangeable; speeds the solver) |

$\hat d_i$ = planned operating minutes of case $i$; $b_s$ = minutes surgeon $s$ is already busy (emergencies, started cases).

**Stage 2 — clock times.** The MILP decides room and surgeon; a sequencing step then sets start times. Inside each room cases run shortest-predicted-first, and a case starts only when its room **and** its surgeon are free and no emergency is in the way. So **no room and no surgeon is ever booked twice at the same time (Constraint 3)**; Step 4 counts clashes and shows the number. Emergencies are not variables: they keep their arrival time and take the first free room; a re-solve of the remaining cases is the same model with the started cases fixed (**Spec 3**, **IS2**).

Variability (**Constraint 2**) enters through $p_i$: the prediction carries a 15 % error (Spec 2), and the simulation in Step 1 draws actual durations from department-specific lognormal distributions, so the plan is judged on what happens, not on what was planned.

## Step 1 · Upload your booking sheet

One file (.xlsx or .csv) with these columns (this is the sheet the OR coordinator already keeps):

| column | example | meaning |
|---|---|---|
| case_id | C00123 | any unique label |
| surgery_date | 2026-01-04 | day of surgery |
| department | General surgery | surgical department |
| surgeon_id | SUR004 | surgeon code |
| room | OR1 | operating room |
| booked_start | 08:00 | planned start time |
| booked_duration_min | 90 | planned length, minutes |

Optional: `is_emergency` (0/1), `booking_date`, `skill_required` (standard/advanced).

No file yet? Run the cell anyway and pick **example** to use a 5-day sample.

In [ ]:
from google.colab import files
import io

choice = input("Upload your own sheet, or use the example?  (type: upload / example) ").strip().lower()
if choice.startswith("u"):
    up = files.upload()                      # file picker opens — .csv or .xlsx
    name, data = list(up.items())[0]
    booking = pd.read_excel(io.BytesIO(data), sheet_name=0) if name.lower().endswith((".xlsx", ".xls")) else pd.read_csv(io.BytesIO(data))
    for col in ("surgery_date", "booking_date"):
        if col in booking: booking[col] = pd.to_datetime(booking[col]).dt.strftime("%Y-%m-%d")
    if "booked_start" in booking: booking["booked_start"] = booking["booked_start"].astype(str).str[:5]
else:
    if not __import__('os').path.exists('sorrs_cases_v2.csv'):
        print("Please upload sorrs_cases_v2.csv (the project dataset) to build the example:"); up = files.upload()
    booking = make_template_from("sorrs_cases_v2.csv", days=5)
    print("Using a 5-day example booking sheet.")

missing = [c for c in TEMPLATE[:7] if c not in booking.columns]
assert not missing, f"Your sheet is missing columns: {missing}"
print(f"\n{len(booking)} cases over {booking.surgery_date.nunique()} days in {booking.room.nunique()} rooms.")

simulated = simulate(booking)
simulated.to_csv("sorrs_simulated.csv", index=False)
files.download("sorrs_simulated.csv")
print("\nDone. sorrs_simulated.csv has your sheet plus: actual start, actual duration, turnover, overrun, waiting days.")
simulated[["case_id","surgery_date","room","booked_start","booked_duration_min","actual_start","actual_duration_min","overrun_min"]].head(8)

## Step 2 · Optimise?

SORRS re-plans every day: which room, which surgeon, what order — keeping rooms inside their 10-hour day, surgeons inside their hours, and work shared between shifts.

Optional: upload your staff roster first (`staff_id, role, department, skill, shift`) so surgeon shifts and skills are exact. Without it, the tool infers them from the booking sheet.

In [ ]:
go = input("Optimise this schedule with SORRS?  (yes / no) ").strip().lower()
if go.startswith("y"):
    r = input("Upload a staff roster? (yes / no) ").strip().lower()
    roster = None
    if r.startswith("y"):
        up = files.upload(); roster = pd.read_csv(io.BytesIO(list(up.values())[0])).set_index("staff_id")
        roster = roster[roster.role == "surgeon"]
    optimized, solve_s = optimize(simulated, roster)
    optimized.to_csv("sorrs_optimized.csv", index=False)
    files.download("sorrs_optimized.csv")
    compare(simulated, optimized, solve_s, roster)
    print("\nsorrs_optimized.csv is ready — open the SORRS dashboard and choose 'Load schedule CSV'.")
else:
    print("Stopped after Step 1. Your simulated sheet is saved as sorrs_simulated.csv.")

## Step 3 · Something changed during the day

The OR coordinator types **five things** — that is all the tool needs:

| what | example |
|---|---|
| date | 2026-01-04 |
| time now | 10:30 |
| department | General surgery |
| surgeon | SUR004 |
| expected minutes | 90 |
| emergency? | yes / no |

What happens: every case that has already started is kept exactly where it is. Each room is treated as free from the end of its current case. An emergency goes into the first free room now. Everything not yet started is re-solved with the same model — a second or two — and a new `sorrs_replanned.csv` is produced for the dashboard.

In [ ]:
assert 'optimized' in globals(), "Run Step 2 first."
print("Dates on this sheet:", ", ".join(sorted(optimized.surgery_date.unique())))
print("Departments:", ", ".join(sorted(optimized.department.unique())))
date   = input("Date (YYYY-MM-DD): ").strip()
now    = input("Time now (HH:MM): ").strip()
dept   = input("Department: ").strip()
surg   = input("Surgeon code (e.g. SUR004): ").strip()
mins   = int(input("Expected duration, minutes: ").strip())
emerg  = input("Emergency? (yes / no): ").strip().lower().startswith("y")

new_case = dict(case_id=f"NEW-{now.replace(':','')}", department=dept, surgeon_id=surg, booked_duration_min=mins, is_emergency=int(emerg))
replanned = replan(optimized, date, now, new_case, roster if 'roster' in globals() else None)
replanned.to_csv("sorrs_replanned.csv", index=False)
files.download("sorrs_replanned.csv")

day = replanned[replanned.surgery_date == date]
day[["case_id","room","surgeon_id","planned_start","planned_duration_min","status"]].sort_values(["room","planned_start"]).reset_index(drop=True)

## Step 4 · Constraint & specification check
Result depends on the booking sheet you uploaded — re-run after a new upload or a re-plan.

In [ ]:
# ---------- Constraint & specification attainment for THIS booking sheet
from IPython.display import Markdown, display
assert 'optimized' in globals(), "Run Step 2 first (answer yes)."
if 'replanned' in globals(): optimized = replanned   # check the latest schedule
sched = optimized[optimized.status != "deferred to next day"]
elect = sched[sched.is_emergency == 0]            # the MILP plans electives; emergencies keep their arrival slot
m = day_summary(simulated.assign(planned_start_min=simulated.booked_start_min), "actual_start_min", "actual_end_min", "manual").mean(numeric_only=True)
o = day_summary(sched, "actual_start_min", "actual_end_min", "SORRS").mean(numeric_only=True)
R_ = roster if 'roster' in globals() and roster is not None else roster_from(simulated)
def gap(df):
    sh = df.surgeon_id.map(R_["shift"]).fillna("A"); a = df[sh=="A"].actual_duration_min.sum(); b = df[sh=="B"].actual_duration_min.sum()
    return abs(a-b)/(a+b) if a+b else 0
g_m = np.mean([gap(g) for _, g in simulated.groupby("surgery_date")]); g_o = np.mean([gap(g) for _, g in sched.groupby("surgery_date")])
ot = 100*(m.overtime_cost_sar-o.overtime_cost_sar)/m.overtime_cost_sar if m.overtime_cost_sar else 0
rh = 100*(m.room_hours-o.room_hours)/m.room_hours; dl = 100*(m.start_delay_min-o.start_delay_min)/m.start_delay_min if m.start_delay_min else 0
room_load = elect.groupby(["surgery_date","room"]).apply(lambda g:(g.planned_duration_min+g.turnover_min).sum()); room_ok = room_load.max() <= CAP+OMAX
surg_load = sched.groupby(["surgery_date","surgeon_id"]).planned_duration_min.sum(); surg_ok = surg_load.max() <= CAP
clash = overlaps(sched)
skill_ok = all((r.skill_required=="standard") or (R_.loc[r.surgeon_id,"skill"]=="advanced") for r in sched[sched.is_emergency==0].itertuples() if r.surgeon_id in R_.index)
n_days = simulated.surgery_date.nunique(); n_rooms = simulated.room.nunique()
defer_pct = 100*(optimized.status=="deferred to next day").mean()
booked_h = (simulated.booked_duration_min.sum()/60)/(n_days*n_rooms)
rows = [
 ("Constraint 1", "Real patient data unavailable (PDPL)", True, "tool runs on the hospital's own booking sheet; no identifiers are read"),
 ("Constraint 2", "Duration variability & emergencies", True, f"actual durations simulated per department; {int(simulated.is_emergency.sum())} emergencies kept at arrival time"),
 ("Constraint 3", "6 rooms × 10 h, no room or surgeon double-booked", room_ok and surg_ok and skill_ok and clash == (0, 0), f"{n_rooms} rooms; clashes in the plan (room, surgeon) = {clash}; max room load {room_load.max():.0f} ≤ {CAP+OMAX} min; max surgeon day {surg_load.max():.0f} ≤ {CAP} min; skill & department matched: {skill_ok}"),
 ("Constraint 4", "Overtime at fixed policy rate", True, f"{TEAM_RATE_SAR_MIN} SAR per overrun minute (1.5 × loaded team rate)"),
 ("Spec 1", "Shift workload gap ≤ 15 %", g_o <= .15, f"{100*g_o:.0f} % with SORRS (was {100*g_m:.0f} %)"),
 ("Spec 3", "Re-schedule ≤ 5 min", solve_s.max() <= 300, f"slowest day {solve_s.max():.1f} s"),
 ("Spec 4", "Room-hours ≤ 85 % of baseline", rh >= 15, f"{rh:+.0f} %" + (f" — comes from {defer_pct:.0f} % of cases moved to the next day" if defer_pct > 5 else " — surgery time is fixed; the saving shows in overtime, not room-hours")),
 ("Spec 5", "Overtime cost −20 %", ot >= 20, f"{ot:+.0f} %  ({m.overtime_cost_sar:,.0f} → {o.overtime_cost_sar:,.0f} SAR/day)"),
 ("Integrated 1", "Waiting −30 % (day-of start delay)", dl >= 30, f"{dl:+.0f} %  ({m.start_delay_min:.0f} → {o.start_delay_min:.0f} min)"),
 ("Integrated 2", "Emergency recovery ≤ 10 min", solve_s.max() <= 600, "solver time plus dashboard refresh"),
 ("Integrated 3", "Room-hours & overtime both −15 %", rh >= 15 and ot >= 15, f"overtime {ot:+.0f} %, room-hours {rh:+.0f} %"),
]
met = sum(r[2] for r in rows)
md = f"### Attainment for this booking sheet — {n_days} day(s), {n_rooms} rooms, {len(simulated)} cases\n\n**{met} of {len(rows)} met** · booked load {booked_h:.1f} h per room-day against a 10 h day · cases done per day {len(sched)/n_days:.1f} (manual {len(simulated)/n_days:.1f}) · **{defer_pct:.0f} % moved to the next day**\n\n"
if defer_pct > 5: md += "> This sheet is overbooked: the model keeps the day feasible by deferring cases. Those patients are rescheduled, not lost — the room-hour figures below reflect that.\n\n"
md += "| | | result | evidence |\n|---|---|---|---|\n"
for k, name, ok, ev in rows: md += f"| **{k}** | {name} | **{'MET' if ok else 'NOT MET'}** | {ev} |\n"
md += "\nSpec 2 (prediction accuracy) and Spec 6 (payback) are evaluated by the ICS and ACFN components, not by this tool."
display(Markdown(md))